# Data pre-processing

In [ ]:
# Step 1: Install libraries
!pip install transformers datasets scikit-learn pandas numpy

In [ ]:
# Step 2: Import essentials
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import BertTokenizerFast

In [ ]:
df = pd.read_csv("/content/imdb_urdu_reviews_test.csv")

In [ ]:
df.head()

,review,sentiment
0,یہ بے گھر خواتین کے بارے میں ایک دستاویزی فلم ...,negative
1,بالکل بھی اچھ ،ی کام نہیں کیا گیا ، پوری فلم ص...,negative
2,یہ عجیب بات ہے کہ کچھ لوگوں کا کیا حشر ہوتا ہے...,negative
3,اور یہ خاص طور پر وکیلوں اور پولیس اہلکاروں کے...,positive
4,پہلے ، ایک وضاحت: میری سرخی کے باوجود ، میں اس...,positive


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5082
negative,4918


In [ ]:
# balancing dataset
from sklearn.utils import resample
import pandas as pd

# Separate classes
df_pos = df[df['sentiment'] == 'positive']
df_neg = df[df['sentiment'] == 'negative']

# Find majority class size
max_size = max(len(df_neg), len(df_pos))

# Oversample minority classes
df_pos_upsampled = resample(df_pos, replace=True, n_samples=max_size, random_state=42)
df_neg_upsampled = resample(df_neg, replace=True, n_samples=max_size, random_state=42)

# Combine and shuffle
df_balanced = pd.concat([df_pos_upsampled, df_neg_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(df_balanced['sentiment'].value_counts())

sentiment
positive    5082
negative    5082
Name: count, dtype: int64


In [ ]:
df.head()

,review,sentiment
0,یہ بے گھر خواتین کے بارے میں ایک دستاویزی فلم ...,negative
1,بالکل بھی اچھ ،ی کام نہیں کیا گیا ، پوری فلم ص...,negative
2,یہ عجیب بات ہے کہ کچھ لوگوں کا کیا حشر ہوتا ہے...,negative
3,اور یہ خاص طور پر وکیلوں اور پولیس اہلکاروں کے...,positive
4,پہلے ، ایک وضاحت: میری سرخی کے باوجود ، میں اس...,positive


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Rename column from 'Class' to 'label'
df_balanced = df_balanced.rename(columns={"sentiment": "label"})
df_balanced = df_balanced.rename(columns={"review": "text"})
# Map labels to integers (P -> 0, N -> 1)
label_map = {'positive': 0, 'negative': 1}
df_balanced['label'] = df_balanced['label'].map(label_map)

# Shuffle before splitting
df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10164 entries, 0 to 10163
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    10164 non-null  object
 1   label   10164 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 158.9+ KB


In [ ]:
print(df.head())

                                                text  label
0  بہتر پوکیمون فلموں میں سے ایک نہیں۔ دو افسانوی...      1
1  اگر آپ اب تک کی بدترین فلم دیکھنا چاہتے ہیں تو...      1
2  اگر میری دادی نے فلمیں کیں تو شاید وہ اس سے کہ...      1
3  یہ فلم ایک ایسی فلم ہے جس نے سن 1932 میں بہت ا...      0
4  لوگوں کے بارے میں ایک عمدہ فلم۔ میں نے چار دوس...      0


In [ ]:
df['label'].value_counts()

,count
label,
1,5082
0,5082


In [ ]:
# Split into train (80%), validation (10%), test (10%)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# Print sizes
print("✅ Train:", len(train_df), " Validation:", len(val_df), " Test:", len(test_df))
print(train_df['label'].value_counts())

✅ Train: 8131  Validation: 1016  Test: 1017
label
1    4066
0    4065
Name: count, dtype: int64


In [ ]:
# Step: Tokenization using Hugging Face Datasets + Tokenizer
# Install first if needed: !pip install transformers datasets

from datasets import Dataset
from transformers import AutoTokenizer

In [ ]:
# 1) Use multilingual BERT (best for Urdu/Roman Urdu + English)
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [ ]:
# 2) Convert pandas DataFrames to Hugging Face Datasets
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

In [ ]:
# 3) Tokenize function
MAX_LEN = 128  # adjust if your sentences are long
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",   # ensures uniform length
        truncation=True,
        max_length=MAX_LEN
    )

In [ ]:
# 4) Apply tokenization
train_ds = train_ds.map(tokenize_function, batched=True)
val_ds   = val_ds.map(tokenize_function, batched=True)
test_ds  = test_ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/8131 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

Map:   0%|          | 0/1017 [00:00<?, ? examples/s]

In [ ]:
# 5) Set format for PyTorch
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

print("✅ Tokenization done.")
print(train_ds[0])

✅ Tokenization done.
{'label': tensor(1), 'input_ids': tensor([   101,    119,    119,    119,  11363,    787,  28971,  34821,    766,
         68595,  10700,  11503,    764,  52437,  10429, 109591,  10429,  10861,
           837,  12427,    770,  10564,  11326,    816,  34084,  11687,  11326,
         13879,  40302,  29869,  10582,  13244,  22848,  23856,  10916,    829,
         20451,  11711,  50888,  52306,    789,  47758,  10429,  11076,  94656,
         12574,  32326,  27416,  10916,  55304,  46541,    752,  11722,  10691,
         92690,  45099,  13185,  12190,    817,  25908,  32219,  29473,    752,
         11722,  10691,  80942,  12190,    816,  14495,  53219,  11503,  13141,
         11091,    819,  13154,  30393,  13044,  42025,  11722,  11503,  40218,
         70664,    784,  34811,  20606,  43632,    818,  10429,  15714,    775,
         56057,    752,    788,  11145,  11242,  11711,    773,  80173,  15896,
         12009,    752,  11363,  55335, 108874,  22147,    763,  

# Full Fine Tuning

In [ ]:
# Install first if needed: !pip install transformers evaluate accelerate
!pip install evaluate

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [ ]:
# 1️⃣ Load model (3 sentiment labels)
MODEL_NAME = "bert-base-multilingual-cased"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 2️⃣ Metric (accuracy)
metric = evaluate.load("accuracy")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

/tmp/ipython-input-2462303314.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # Evaluation + saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # Training settings
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,            # standard for full BERT fine-tuning
    warmup_ratio=0.1,              # improves accuracy & stabilizes training
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),

    # Logging
    logging_dir="./logs",
    logging_steps=50,

    # Enable Weights & Biases
    report_to="wandb",             # <--- ENABLE W&B
    run_name="bert_full_finetune", # your run name

    # Reproducibility
    seed=42,
)


In [ ]:
# 4️⃣ Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-705311372.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 5️⃣ Train
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adeelahmed4868 (adeelahmed4868-riphah-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.545000,0.535150,0.743110
2,0.430000,0.565123,0.782480
3,0.338700,0.371151,0.862205
4,0.228700,0.445473,0.879921
5,0.207600,0.563797,0.874016
6,0.068000,0.638497,0.882874
7,0.052600,0.661962,0.885827
8,0.054800,0.678505,0.889764
9,0.026200,0.679164,0.892717
10,0.024300,0.718983,0.889764


TrainOutput(global_step=5090, training_loss=0.20189761147049415, metrics={'train_runtime': 1888.0238, 'train_samples_per_second': 43.066, 'train_steps_per_second': 2.696, 'total_flos': 5348389977830400.0, 'train_loss': 0.20189761147049415, 'epoch': 10.0})

In [ ]:
trainer.predict(test_ds)

PredictionOutput(predictions=array([[-4.4101562,  3.3847656],
       [ 4.0664062, -2.765625 ],
       [-4.4140625,  3.3964844],
       ...,
       [-4.4375   ,  3.3222656],
       [ 4.0273438, -2.6796875],
       [-4.1171875,  2.90625  ]], dtype=float32), label_ids=array([1, 0, 1, ..., 1, 0, 0]), metrics={'test_loss': 0.6167380809783936, 'test_accuracy': 0.9016715830875123, 'test_runtime': 2.0465, 'test_samples_per_second': 496.95, 'test_steps_per_second': 31.273})

# LORA

In [ ]:
pip install transformers datasets peft accelerate bitsandbytes evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.9 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import BertForSequenceClassification, BertTokenizerFast, TrainingArguments, Trainer, BitsAndBytesConfig
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import evaluate

# ---------------------------
# 1. Load tokenizer and model
# ---------------------------
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizerFast.from_pretrained(model_name)


model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,           # <-- change for your dataset
    device_map="auto"
)

# ---------------------------
# 2. Prepare LoRA configuration


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "output.dense"]  # strongest effect
)


model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ---------------------------
# 3. Load dataset
# ---------------------------
# dataset = pd.read_csv("/content/combined.csv")   # example dataset; replace with yours
# The dataset loading and preparation has been done in previous cells.
# Use the already prepared datasets from previous steps.
train_set = train_ds # Changed from train_df
test_set = test_ds   # Changed from test_df

# ---------------------------
# 4. Tokenization
# ---------------------------
# Tokenization is already handled in previous cells and train_ds, test_ds are ready.

# ---------------------------
# 5. Metric — Accuracy
# ---------------------------
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# ---------------------------
# 6. Training Arguments

training_args = TrainingArguments(
    output_dir="./mb_lora",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,               # LoRA needs higher LR
    num_train_epochs=15,

    fp16=True,

    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,

    weight_decay=0.01,                # ★ improves generalization
    warmup_ratio=0.1,                 # ★ stabilizes training
    lr_scheduler_type="cosine",       # ★ smoother LR → better accuracy

    logging_steps=50,
    seed=42,

    report_to="wandb",                # optional: enable W&B logging
    run_name="bert_lora_optimized"
)

# ---------------------------
# 7. Trainer
# ---------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# ---------------------------
# 8. Train
# ---------------------------
trainer.train()

# ---------------------------
# 9. Evaluate
# ---------------------------
results = trainer.evaluate()
print("Final Accuracy:", results["eval_accuracy"])


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,918,466 || all params: 179,773,444 || trainable%: 1.0672


/tmp/ipython-input-1214348026.py:94: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.622900,0.615587,0.665683
2,0.520500,0.606769,0.737463
3,0.389100,0.408194,0.825959
4,0.315200,0.377905,0.855457
5,0.267800,0.482916,0.856441
6,0.185800,0.427131,0.885939
7,0.173900,0.517729,0.880039
8,0.107400,0.536700,0.897738
9,0.087100,0.711646,0.887906
10,0.033900,0.660028,0.901672


Final Accuracy: 0.855457227138643


In [ ]:
trainer.predict(test_ds)

PredictionOutput(predictions=array([[-2.6542969 ,  2.6582031 ],
       [ 1.5683594 , -1.9648438 ],
       [-2.3984375 ,  2.4179688 ],
       ...,
       [ 1.28125   , -1.6337891 ],
       [-0.3251953 ,  0.12304688],
       [-1.6855469 ,  1.6044922 ]], dtype=float32), label_ids=array([1, 0, 1, ..., 1, 0, 0]), metrics={'test_loss': 0.37790507078170776, 'test_accuracy': 0.855457227138643, 'test_runtime': 10.2026, 'test_samples_per_second': 99.681, 'test_steps_per_second': 12.546})

In [ ]:
# previously used
# LORA configuration--------------------------
lora_config = LoraConfig(
    r=8,                    # LoRA rank (4–16 normal)
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
# training arguments---------------------------
training_args = TrainingArguments(
    output_dir="./mb_lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   # helps train on small GPUs
    learning_rate=2e-4,              # higher LR for LoRA
    num_train_epochs=12,
    fp16=True,                       # mixed precision for speed
    eval_strategy="epoch",           # Changed from evaluation_strategy
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True
)

# Output
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
trainable params: 296,450 || all params: 178,151,428 || trainable%: 0.1664
/tmp/ipython-input-50743728.py:78: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
 [4072/4072 08:48, Epoch 8/8]
Epoch	Training Loss	Validation Loss	Accuracy
1	0.630600	0.608878	0.658800
2	0.560400	0.572287	0.698132
3	0.480100	0.490803	0.769912
4	0.440600	0.544214	0.739430
5	0.418800	0.465649	0.774828
6	0.381600	0.449205	0.794494
7	0.356200	0.455079	0.816126
8	0.355200	0.451518	0.814159
 [128/128 00:02]
Final Accuracy: 0.7944936086529006

NameError: name 'LoraConfig' is not defined

# Prefix Tuning (P-Tuning v2) for BERT

In [ ]:
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


In [ ]:
#  Configure Prefix Tuning
from peft import PrefixTuningConfig, get_peft_model, TaskType

# Optimized Prefix Tuning configuration
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,  # sequence classification
    num_virtual_tokens=40,       # number of prefix tokens
    prefix_projection=True,      # improves representation learning
    inference_mode=False,
    encoder_hidden_size=768      # explicitly set the hidden size for projection
)


In [ ]:
from transformers import AutoModelForSequenceClassification

# Load a fresh base model for Prefix Tuning with the correct number of labels
base_model_for_prefix_tuning = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Apply Prefix Tuning to the fresh base model
model = get_peft_model(base_model_for_prefix_tuning, prefix_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Show trainable parameters
model.print_trainable_parameters()

trainable params: 14,797,058 || all params: 192,652,036 || trainable%: 7.6807


In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./results_prefix",

    # ------ Evaluation / Saving ------
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,     # SAFE now (fix shown below)
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # ------ Hyperparameters ------
    learning_rate=3e-4,              # good for prefix tuning
    num_train_epochs=8,              # prefix tuning benefits from more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_ratio=0.1,
    weight_decay=0.01,

    # ------ Precision ------
    fp16=torch.cuda.is_available(),

    # ------ Logging / WandB ------
    logging_steps=50,
    report_to="wandb",               # ENABLE WandB
    run_name="prefix_tuning_experiment",  # WandB experiment name

    # ------ Reproducibility ------
    seed=42
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-886978032.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 🔥 Train Prefix Tuning model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.698900,0.697004,0.500000
2,0.697000,0.704093,0.500000


KeyboardInterrupt: 

In [ ]:
# Evaluate
results = trainer.evaluate(test_ds)
print("✅ Prefix Tuning Test Results:", results)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.698900,0.697004,0.500000
2,0.695800,0.694497,0.500492


✅ Prefix Tuning Test Results: {'eval_loss': 0.694497287273407, 'eval_accuracy': 0.5004916420845624}


In [ ]:
# prompt tunning

In [ ]:
# Step 1: Load base model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PromptTuningConfig, get_peft_model

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Step 1b: Define prompt tuning configuration
prompt_config = PromptTuningConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=60,                    # more expressive prompts
    tokenizer_name_or_path=model_name,

    # --- IMPROVEMENT: better initialization ---
    prompt_tuning_init="TEXT",
    prompt_tuning_init_text="Classify the sentiment:",
)


In [ ]:
# Step 1c: Create PEFT model
peft_model = get_peft_model(model, prompt_config)
peft_model.print_trainable_parameters()

trainable params: 48,387 || all params: 177,904,134 || trainable%: 0.0272


In [ ]:
training_args = TrainingArguments(
    output_dir="./prompt_tuned_bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    learning_rate=5e-4,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    fp16=True,

    logging_steps=50,
    report_to="wandb",              # 🔥 WandB enabled
    run_name="prompt_tuning_experiment",
    seed=42
)


In [ ]:
import numpy as np
import evaluate

# Load accuracy metric
accuracy = evaluate.load("accuracy")

# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# Create Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-431711176.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.711700,0.695003,0.515748
2,0.697200,0.709288,0.500000
3,0.672500,0.684487,0.571850
4,0.667200,0.687912,0.568898
5,0.659700,0.670294,0.605315
6,0.656600,0.669368,0.604331
7,0.645400,0.667402,0.598425
8,0.647700,0.664631,0.600394


TrainOutput(global_step=2040, training_loss=0.6809694177964154, metrics={'train_runtime': 418.5356, 'train_samples_per_second': 155.418, 'train_steps_per_second': 4.874, 'total_flos': 4278865649577984.0, 'train_loss': 0.6809694177964154, 'epoch': 8.0})

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)

✅ Test Results: {'eval_loss': 0.6499390602111816, 'eval_accuracy': 0.6253687315634219, 'eval_runtime': 2.5117, 'eval_samples_per_second': 404.898, 'eval_steps_per_second': 12.74, 'epoch': 8.0}


# P-Tuning v2

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PromptEncoderConfig, get_peft_model

# Load base model & tokenizer
model_name ="bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Define P-Tuning v2 configuration
peft_config = PromptEncoderConfig(
    task_type="SEQ_CLS",
    num_virtual_tokens=60,     # more expressive prompts → higher accuracy
    encoder_hidden_size=768
)

# Apply PEFT configuration
peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,820,163 || all params: 179,675,910 || trainable%: 1.0130


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# ✅ Training arguments (optimized for P-Tuning v2)
training_args = TrainingArguments(
    output_dir="./p_tuning_v2_results",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,

    learning_rate=3e-4,
    num_train_epochs=8,                  # slightly more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,                    # stabilizes high LR
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    fp16=torch.cuda.is_available(),      # faster + less memory

    logging_dir="./logs",
    logging_steps=50,

    # Enable WandB (optional)
    report_to="wandb",
    run_name="p_tuning_v2_experiment",
    seed=42
)


# ✅ Metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# ✅ Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-1205006206.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# ✅ Step 4: Train and Evaluate P-Tuning v2 model

# Train
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.713000,0.695260,0.497047
2,0.701400,0.737722,0.500000
3,0.692200,0.682537,0.574803
4,0.662200,0.717736,0.505906
5,0.650200,0.670729,0.603346
6,0.638400,0.661343,0.610236
7,0.623800,0.648945,0.620079
8,0.632900,0.652741,0.625000


TrainOutput(global_step=2040, training_loss=0.675881986992032, metrics={'train_runtime': 419.4126, 'train_samples_per_second': 155.093, 'train_steps_per_second': 4.864, 'total_flos': 4278865649577984.0, 'train_loss': 0.675881986992032, 'epoch': 8.0})

In [ ]:
 # Evaluate P-Tuning v2 model

# Evaluate on validation set
val_results = trainer.evaluate()
print("✅ Validation Results:", val_results)

# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print("✅ Test Results:", test_results)


✅ Validation Results: {'eval_loss': 0.7006297707557678, 'eval_accuracy': 0.5600393700787402, 'eval_runtime': 2.4007, 'eval_samples_per_second': 423.207, 'eval_steps_per_second': 13.329, 'epoch': 8.0}
✅ Test Results: {'eval_loss': 0.7002084255218506, 'eval_accuracy': 0.5742379547689282, 'eval_runtime': 2.3836, 'eval_samples_per_second': 426.657, 'eval_steps_per_second': 13.425, 'epoch': 8.0}


# AdaLoRA

In [ ]:
# Run in a notebook cell / terminal
!pip install -q transformers peft accelerate evaluate bitsandbytes datasets


In [ ]:
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "bert-base-multilingual-cased"   # best for Urdu + Roman Urdu + English
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import AdaLoraConfig, get_peft_model, TaskType

adalora_config = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,       # sequence classification
    r=16,                              # initial LoRA rank
    lora_alpha=32,                     # scaling factor
    target_modules=["query", "value"], # adapt attention Q/V layers
    lora_dropout=0.05,                 # low dropout for better accuracy
    inference_mode=False,              # must be False during training
    init_r=16,                         # starting rank for adaptation
    target_r=6,                        # final target low-rank
    tinit=100,                         # steps before adaptation starts
    tfinal=600,                        # steps when adaptation ends
    deltaT=10,                          # adaptation frequency (steps)
    total_step=1000                     # placeholder, calculate from dataset size
)

adalora_model = get_peft_model(base_model, adalora_config)
adalora_model.print_trainable_parameters()

trainable params: 592,515 || all params: 178,448,286 || trainable%: 0.3320


/usr/local/lib/python3.12/dist-packages/peft/tuners/adalora/config.py:96: UserWarning: Note that `r` is not used in AdaLora and will be ignored.If you intended to set the initial rank, use `init_r` instead.
  warnings.warn(


In [ ]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./adalora_results",

    # --- Evaluation & Saving ---
    eval_strategy="epoch",       # evaluate after each epoch
    save_strategy="epoch",             # save checkpoints each epoch
    load_best_model_at_end=True,       # restore best model automatically
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,                # keep only last 2 checkpoints

    # --- Hyperparameters ---
    learning_rate=3e-4,                # recommended for adapter methods like AdaLoRA
    num_train_epochs=8,                # 4–8 works well for small datasets
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,                  # stabilizes early training
    weight_decay=0.01,                 # prevents overfitting
    gradient_accumulation_steps=2,     # optional for simulating larger batch
    fp16=torch.cuda.is_available(),    # mixed precision for speed & memory efficiency

    # --- Misc / Logging ---
    seed=SEED,                         # ensures reproducibility
    dataloader_pin_memory=False,       # avoid pin_memory warnings on CPU
    logging_steps=50,
    report_to="none"                   # disable automatic logging services (WandB, TensorBoard)
)


In [ ]:
from transformers import Trainer
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=adalora_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # stops if no val improvement for 2 epochs
)


/tmp/ipython-input-3692918051.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.707500,0.700845,0.500000
2,0.693200,0.696281,0.500000
3,0.642200,0.643330,0.636811
4,0.610500,0.638605,0.643701
5,0.610300,0.592208,0.690945
6,0.593900,0.592705,0.699803
7,0.583000,0.571961,0.719488
8,0.558400,0.567668,0.719488


TrainOutput(global_step=2040, training_loss=0.72062074062871, metrics={'train_runtime': 483.061, 'train_samples_per_second': 134.658, 'train_steps_per_second': 4.223, 'total_flos': 4308351789330432.0, 'train_loss': 0.72062074062871, 'epoch': 8.0})

In [ ]:
val_results = trainer.evaluate(eval_dataset=val_ds)
print("Validation:", val_results)

test_results = trainer.evaluate(eval_dataset=test_ds)
print("Test:", test_results)


Validation: {'eval_loss': 0.5625829100608826, 'eval_accuracy': 0.7293307086614174, 'eval_runtime': 2.2133, 'eval_samples_per_second': 459.048, 'eval_steps_per_second': 14.458, 'epoch': 6.0}
Test: {'eval_loss': 0.5717098712921143, 'eval_accuracy': 0.7050147492625368, 'eval_runtime': 2.2217, 'eval_samples_per_second': 457.752, 'eval_steps_per_second': 14.403, 'epoch': 6.0}


# IA³ fine-tuning

In [ ]:
from peft import IA3Config, get_peft_model, TaskType

# ---------------------------
# 1️⃣ Define IA³ Configuration
# ---------------------------
ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,          # sequence classification task
    target_modules=["query", "value"],   # apply IA³ adapters to attention Q/V layers
    # feedforward_modules=["intermediate.dense"],  # optional: include feed-forward layers for better adaptation
)

# ---------------------------
# 2️⃣ Apply IA³ to Base Model
# ---------------------------
ia3_model = get_peft_model(base_model, ia3_config)

# ---------------------------
# 3️⃣ Print Trainable Parameters
# ---------------------------
ia3_model.print_trainable_parameters()


trainable params: 20,739 || all params: 178,466,718 || trainable%: 0.0116


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
from peft import IA3Config, get_peft_model, TaskType

ia3_config = IA3Config(
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],  # focus on attention parts for best results
    # feedforward_modules=["intermediate.dense"],  # optional for better adaptation
)

ia3_model = get_peft_model(base_model, ia3_config)
ia3_model.print_trainable_parameters()

trainable params: 20,739 || all params: 178,466,718 || trainable%: 0.0116


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
import evaluate, numpy as np
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ia3_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    learning_rate=2e-4,              # slightly smaller than LoRA for stability
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_total_limit=2,
    dataloader_pin_memory=False,
    seed=42,
    report_to="none"                 # turn off W&B etc.
)

In [ ]:
from transformers import TrainingArguments
import torch

training_args = TrainingArguments(
    output_dir="./ia3_results",

    # --- Evaluation & Saving ---
    eval_strategy="epoch",       # evaluate model at the end of each epoch
    save_strategy="epoch",             # save checkpoints each epoch
    load_best_model_at_end=True,       # restore the best model automatically
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,                # keep only the last 2 checkpoints

    # --- Hyperparameters ---
    learning_rate=2e-4,                # slightly smaller than LoRA for stability
    num_train_epochs=6,                # 4–8 works well for small datasets
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,                  # stabilizes early training
    weight_decay=0.01,                 # prevents overfitting
    gradient_accumulation_steps=2,     # optional: simulates larger batch
    fp16=torch.cuda.is_available(),    # mixed precision for faster training

    # --- Misc / Logging ---
    logging_steps=50,
    dataloader_pin_memory=False,       # avoid warnings on CPU
    seed=42,                            # ensures reproducibility
    report_to="none"                   # disables WandB or other logging
)


In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=ia3_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


/tmp/ipython-input-656194628.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.572000,0.566048,0.725394
2,0.569100,0.572123,0.717520
3,0.534800,0.567030,0.726378
4,0.556800,0.565694,0.727362
5,0.563900,0.565079,0.728346
6,0.560700,0.562583,0.729331


TrainOutput(global_step=1530, training_loss=0.558132232715881, metrics={'train_runtime': 308.5347, 'train_samples_per_second': 158.122, 'train_steps_per_second': 4.959, 'total_flos': 3231954445685760.0, 'train_loss': 0.558132232715881, 'epoch': 6.0})

In [ ]:
val_results = trainer.evaluate(val_ds)
print("✅ Validation:", val_results)

test_results = trainer.evaluate(test_ds)
print("✅ Test:", test_results)

✅ Validation: {'eval_loss': 0.5625829100608826, 'eval_accuracy': 0.7293307086614174, 'eval_runtime': 2.4119, 'eval_samples_per_second': 421.25, 'eval_steps_per_second': 13.268, 'epoch': 6.0}
✅ Test: {'eval_loss': 0.5717098712921143, 'eval_accuracy': 0.7050147492625368, 'eval_runtime': 2.6514, 'eval_samples_per_second': 383.568, 'eval_steps_per_second': 12.069, 'epoch': 6.0}


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.546500,0.557528,0.723425
2,0.564700,0.567925,0.722441
3,0.518400,0.564585,0.725394
4,0.537200,0.555824,0.736220
5,0.558100,0.560497,0.735236
6,0.544900,0.554113,0.733268


TrainOutput(global_step=3054, training_loss=0.5469256868999459, metrics={'train_runtime': 321.8883, 'train_samples_per_second': 151.562, 'train_steps_per_second': 9.488, 'total_flos': 3231954445685760.0, 'train_loss': 0.5469256868999459, 'epoch': 6.0})

In [ ]:
val_results = trainer.evaluate(val_ds)
print("✅ Validation:", val_results)

test_results = trainer.evaluate(test_ds)
print("✅ Test:", test_results)

✅ Validation: {'eval_loss': 0.5558238625526428, 'eval_accuracy': 0.7362204724409449, 'eval_runtime': 2.2547, 'eval_samples_per_second': 450.61, 'eval_steps_per_second': 14.192, 'epoch': 6.0}
✅ Test: {'eval_loss': 0.5658618211746216, 'eval_accuracy': 0.711897738446411, 'eval_runtime': 2.2285, 'eval_samples_per_second': 456.357, 'eval_steps_per_second': 14.359, 'epoch': 6.0}
